In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
from pathlib import Path
import numpy as np

In [19]:


class QuickDrawDataset(Dataset):
    def __init__(self, quickdraw_folder="quickdraw_data", split='train', max_per_class=10000):
        self.data_folder = Path(quickdraw_folder)
        # Exclude .full.npz files to avoid duplicates
        all_files = self.data_folder.glob("*.npz")
        self.files = sorted([f for f in all_files if not f.stem.endswith('.full')])
        self.split = split
        self.max_per_class = max_per_class
        self.label_to_idx = {f.stem: i for i, f in enumerate(self.files)}
        
        # Build index mapping: global_idx -> (file_path, drawing_idx_in_file, label)
        self.index_map = []
        n_files_loaded = 0
        print(len(self.files))
        for file_path in self.files:
            n_files_loaded +=1
            label = file_path.stem
            # Only load metadata to get count, not actual data
            data = np.load(file_path, allow_pickle=True, encoding='latin1')
            num_drawings = min(len(data[split]), self.max_per_class)
            
            # Store references, not actual drawings
            for drawing_idx in range(num_drawings):
                self.index_map.append((file_path, drawing_idx, self.label_to_idx[label]))
            print(n_files_loaded)
            data.close()  # Close the file
        
        print(f"Dataset initialized with {len(self.index_map)} samples from {len(self.files)} classes")
    
    def __len__(self):
        return len(self.index_map)
    
    def __getitem__(self, idx):
        file_path, drawing_idx, label = self.index_map[idx]
        
        # Lazy load: only load the specific drawing when needed
        data = np.load(file_path, allow_pickle=True, encoding='latin1')
        drawing = data[self.split][drawing_idx]
        data.close()
        
        # Convert drawing to tensor
        return torch.tensor(drawing, dtype=torch.float32), label




In [20]:
dataset = QuickDrawDataset(split='train', max_per_class=100)
dataloader = DataLoader(dataset=dataset, batch_size=32, shuffle=True)


345
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276


In [23]:
print(dataset.__getitem__(1))

(tensor([[  59.,   15.,    0.],
        [ 165.,   25.,    0.],
        [ 200.,   -4.,    0.],
        [  52.,   -8.,    0.],
        [  51.,   -1.,    0.],
        [  10.,    9.,    0.],
        [  10.,   18.,    0.],
        [ -21.,   17.,    0.],
        [ -46.,   27.,    0.],
        [ -33.,   26.,    0.],
        [ -49.,   22.,    0.],
        [ -86.,   17.,    0.],
        [ -28.,   -1.,    0.],
        [ -55.,   -9.,    0.],
        [ -99.,   -1.,    0.],
        [ -46.,   -7.,    0.],
        [ -68.,  -19.,    0.],
        [ -21.,  -36.,    0.],
        [  -1.,  -33.,    0.],
        [   8.,  -22.,    0.],
        [  -2.,  -41.,    1.],
        [ 408.,   31.,    0.],
        [  31.,  -65.,    0.],
        [   5.,  -27.,    0.],
        [  -1.,  -54.,    0.],
        [ -16.,  -25.,    0.],
        [   3.,  -33.,    0.],
        [  -7.,  -10.,    0.],
        [  62.,    1.,    1.],
        [   0.,   -4.,    0.],
        [  48.,   18.,    0.],
        [   0.,  115.,    0.],
       